# Bias detection in classification

This is a gap-fill version of our applications to fairness tutorial given to FAccT. It was adapted for teaching purposes.

The usecase is the BIOS dataset, where biographies of people are classified into occupations. Our goal will be to detect if there is any gender bias in the model classification. For that we will use `interpreto`.

There are several parts to this notebook:

0. Setting up the environment, loading the dataset and the model
1. Attributions
2. Probing the gender concept
3. Unsupervised concepts analysis

*authors: Antonin Poché & Fanny Jourdan*

## 0. Setup

### 0.1 Setting up the environment

We use `uv` because it is faster, but you can use `pip`.

In [ ]:
# create the env
# uv venv

# activate the env
# source .venv/bin/activate  # linux
# .venv/Scripts/activate  # windows

# install required packages
# !uv pip install datasets interpreto

In [ ]:
# verify our installation
import datasets
import interpreto
import torch
import transformers

In [ ]:
if not torch.cuda.is_available():
    print("Warning: CUDA is not available. This may become quite long, we recommend tuning down the parameters.")
    device = "cpu"
else:
    device = "cuda"

### 0.2 Loading the dataset

We use the [BIOS dataset](https://huggingface.co/datasets/LabHC/bias_in_bios) from HuggingFace.

Paper: [De-Artega et al. - FAccT 2019 - A Case Study of Semantic Representation Bias in a High-Stakes Setting](https://doi.org/10.1145/3287560.3287572)

In [ ]:
dataset = datasets.load_dataset("LabHC/bias_in_bios")
classes_names = [
    "accountant",
    "architect",
    "attorney",
    "chiropractor",
    "comedian",
    "composer",
    "dentist",
    "dietitian",
    "dj",
    "filmmaker",
    "interior_designer",
    "journalist",
    "model",
    "nurse",
    "painter",
    "paralegal",
    "pastor",
    "personal_trainer",
    "photographer",
    "physician",
    "poet",
    "professor",
    "psychologist",
    "rapper",
    "software_engineer",
    "surgeon",
    "teacher",
    "yoga_teacher"
]
gender_names = ["male", "female"]

### 0.3 Loading the model

Choose between `"Fannyjrd/roberta-bios"` and `"Fannyjrd/roberta-bios-biased"`. They are both roberta for sequence classification.

This way you can compare if explanations indeed help detecting biases.

In [ ]:

from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "Fannyjrd/roberta-bios-biased"  # or "Fannyjrd/roberta-bios"
model = AutoModelForSequenceClassification.from_pretrained(model_name, device_map=device)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)

## 1. Attributions

Attribution highlight the most important inputs for a given prediction.

### 1.1 Find representative examples

We cannot check explanations for all examples, so we need to choose a few representative examples. In theory we should cover several elements of each class with both errors and correct predictions.

In practice, let's focus on few errors. You are free to explore on your own.

In [ ]:
studied_ids = [107, 155, 1062]
studied_preds = [10, 13, 13]

studied_samples = dataset["test"].select(studied_ids)

studied_texts = list(studied_samples["hard_text"])

### 1.2 Compute an attributions

Choose an [attribution method](https://for-sight-ai.github.io/interpreto/api/attributions/overview/) and compute the attributions for the studied examples.

In [ ]:
from interpreto import Lime, plot_attributions

# Construct the explainer
explainer = Lime(model, tokenizer, n_perturbations=200)

# Explain both the label and prediction logits for each studied samples
attributions = {}
targets = torch.stack([torch.tensor(studied_samples["profession"]), torch.tensor(studied_preds)]).T
attributions["Lime"] = explainer.explain(studied_texts, targets)

for a in attributions["Lime"]:
    plot_attributions(a, classes_names=classes_names)

### 1.3 Check with other methods

There is no method to rule them all. It is important to check with several methods to see if the results are consistent. Explore with other ones. At least a perturbation-based and a gradient-based method should be used.

In [ ]:
from interpreto import IntegratedGradients, Occlusion

for method in [IntegratedGradients, Occlusion]:
    print(f"===================\n{method.__name__}\n===================")
    explainer = method(model, tokenizer)
    attributions[method.__name__] = explainer.explain(studied_texts, targets)
    for a in attributions[method.__name__]:
        plot_attributions(a, classes_names=classes_names)

### 1.4 Apply metrics

Interpreto proposes the [`Deletion`](https://for-sight-ai.github.io/interpreto/api/attributions/metrics/deletion/) and [`Insertion`](https://for-sight-ai.github.io/interpreto/api/attributions/metrics/insertion/) metrics to evaluate the faithfulness of attributions. The idea is to remove or add tokens in their order of importance and check if it impacts the output coherently.

Note that in practice, the more samples you have the more robust will the score be.

Paper: [Petsiuk et al. - ICLR 2018 - RISE: Randomized Input Sampling for Explanation of Black-box Models](https://arxiv.org/abs/1806.07421)

In [ ]:
from interpreto.attributions.metrics import Deletion

metric = Deletion(model, tokenizer)

for method_name, attribution in attributions.items():
    score,_ = metric.evaluate(attribution)
    print(f"{method_name} deletion score: {score:.4f}")

### 1.5 Going further: global explanations

For a given sample, we listed the most important words.

But what are the most important words for a given class?

Compute many attributions and for each class count the number of times a word appears in the top-3 most important words. Then list the top-10 most important words for each class.

In [ ]:
...  # TODO

## 2. Probing the gender concept



### 2.1 Split the model and compute activations

We split the model between the encoder $enc$ and the classification head $cls$ thanks to the `splitter`. Then compute the activations $z$ as the output of the encoder (they correspond to the [CLS] token).

$f = cls \circ enc$

$z = enc(x)$

For this, please rely on the [`SplitterForClassification`](https://for-sight-ai.github.io/interpreto/api/concepts/splitters/splitter_for_classification/) from interpreto and its [`get_activations()`](https://for-sight-ai.github.io/interpreto/api/concepts/splitters/splitter_for_classification/#interpreto.SplitterForClassification.get_activations) method.

In [ ]:
from interpreto import SplitterForClassification

# Get inputs and gender label from the dataset
train_inputs = list(dataset["train"].shuffle(seed=42).select(range(1000))["hard_text"])
test_inputs = list(dataset["test"].shuffle(seed=42).select(range(1000))["hard_text"])

# instantiate the splitter from the model
splitter = SplitterForClassification(
    model,
    tokenizer=tokenizer,
    device_map=device,
    batch_size=32,
)

# compute their activations
train_acts, train_occ_preds = splitter.get_activations(train_inputs, tqdm_bar=True, forward_kwargs={"truncation": True})
test_acts, test_occ_preds = splitter.get_activations(test_inputs, tqdm_bar=True, forward_kwargs={"truncation": True})

### 2.2 Train a linear probe to predict gender from activations (CAV)

We suggest using a linear SVM as in the original CAV (Concept Activation Vector) paper [1]. Interpreto proposes the [`LinearSVMProbe`](https://for-sight-ai.github.io/interpreto/api/concepts/probes/#interpreto.concepts.probes.LinearSVMProbe) that you can wrap with the [`ProbeExplainer`](https://for-sight-ai.github.io/interpreto/api/concepts/probes/#interpreto.concepts.probes.ProbeExplainer) to link it with the splitter. Do not hesitate to include a `l2` regularization.

$$
s(z) = w^T z + b
$$

[1] [Kim et al. - ICML 2018 - Interpretability Beyond Feature Attribution: Quantitative Testing with Concept Activation Vectors (TCAV)](https://arxiv.org/abs/1711.11279)

In [ ]:
from interpreto.concepts.probes import LinearSVMProbe, ProbeExplainer

train_genders = torch.tensor(dataset["train"].shuffle(seed=42).select(range(1000))["gender"])
test_genders = torch.tensor(dataset["test"].shuffle(seed=42).select(range(1000))["gender"])

# Creating the probe model and linking it with the splitter
probe = LinearSVMProbe(lr=1e-2, max_iter=200, normalization=None)
probe_explainer = ProbeExplainer(splitter, probe)
probe_explainer.to(device)

# Training the probe classifier
probe_explainer.fit(train_acts.to(device), train_genders.float().unsqueeze(1).to(device))

We then test the probe's performance to see if the gender concept is linearly accessible from the model's activations.

In [ ]:
# Validating the probe performance
with torch.no_grad():
    test_scores = probe_explainer.activations_to_concepts(test_acts.to(device))
test_gender_preds = (test_scores.squeeze(1) > 0).long()
accuracy = (test_gender_preds == test_genders.cuda()).float().mean().item()
print(f"test gender probe accuracy: {accuracy:.3f}")

### 2.3 Compute the TCAV score

TCAV for Testing with CAVs check if the concept was used in the prediction. It checks sensibility of the model outputs to the concept direction.

CAV: $w$

TCAV: $\nabla_z cls(z) \cdot w$

This is not directly implemented in interpreto, so we rely on the splitter's `_get_concept_output_gradients` to obtain the gradients. But it is a work around, future versions will allow simpler solutions for this.

In [ ]:
import torch.nn.functional as F

class TCAV():
    threshold: None | float = None

    def __init__(self, explainer: ProbeExplainer):
        cav = explainer.concept_model.weight.detach().squeeze()
        self.cav = cav / cav.norm()
        self.splitter = explainer.splitter

    def gradients(self, inputs: list[str] | torch.Tensor) -> torch.Tensor:
        """
        Compute the gradients using the splitter methods.

        Args:
            inputs:
                (list[str]): The text samples or
                (torch.Tensor): Their activations. (n, d)
        Returns:
            (torch.Tensor): The gradients. (n, cls,d)
        """
        gradients_list = splitter._get_concept_output_gradients(
            inputs=inputs,
            activations_to_concepts=lambda z: z,  # placeholder
            concepts_to_activations=lambda z: z,  # placeholder
            concepts_x_gradients=False,
        )
        return torch.stack(gradients_list).squeeze().cuda()

    def calibrate(self, gradients: torch.Tensor, percentile: float = 0.95, sampling: int = 100):
        """
        Fixes a threshold to later determine if directions are meaningful.
        The threshold is based on randomly sampled directions.

        Args:
            gradients: The gradients of the outputs with respects to the activations (n, cls, d)
            percentile: CAVs are deamned meaningful if their absolute score is greater than {percentile}.
            sampling: The number of random directions to sample.
        """
        torch.manual_seed(0)
        random_directions = F.normalize(torch.randn(self.cav.shape[0], sampling, device=device), dim=0)  # (d, s)
        t_random = gradients @ random_directions  # (n, cls, s)
        self.threshold = torch.quantile(t_random.abs().permute(1, 0, 2).flatten(1, 2), 0.95, dim=-1)  # (cls,)

    def __call__(self, gradients: torch.Tensor) -> torch.Tensor:
        """
        Args:
            gradients: The gradients of the activations to outputs function with respect to the activations (n, cls, d).
        Returns:
            (torch.Tensor): The TCAV scores. (n, cls)
        """
        return gradients @ self.cav

In [ ]:
# Construct and calibrate TCAV
tcav = TCAV(probe_explainer)
gradients = tcav.gradients(test_inputs)  # We need a certain number for calibration
tcav.calibrate(gradients)

# Compute scores on our studied samples
studied_gradients = tcav.gradients(studied_texts)
for sample, pred, grad in zip(studied_texts, studied_preds, studied_gradients):
    score = tcav(grad)[pred]
    feminine_impact = (score > tcav.threshold[pred]).item()
    print(f"Sample: {sample}\nPredicted class: {classes_names[pred]}, TCAV score: {score.item():.4f}, Feminine impact: {feminine_impact}\n")

### 2.4 Going further: Global classes sensitivity to gender

We were able to measure the gender concept impact on the model's predictions. But we can go further and measure the impact of gender on each class.

This means computing the gender impact on the model's predictions for each sample and counting the proportions of samples impacted in each class.

In [ ]:
test_scores = tcav(gradients)

# aggregate gender sensitivity between masculine/neutral/feminine
masc_rates = (test_scores <= -tcav.threshold).float().mean(0)
fem_rates = (test_scores >= tcav.threshold).float().mean(0)
neutral_rates = 1 - masc_rates - fem_rates

In [ ]:
import matplotlib.pyplot as plt

# Sort by net direction
score = fem_rates - masc_rates
order = torch.argsort(score)

masc = masc_rates[order].detach().cpu()
neutral = neutral_rates[order].detach().cpu()
fem = fem_rates[order].detach().cpu()
names = [classes_names[i.item()] for i in order]

y = torch.arange(len(names))

fig, ax = plt.subplots(figsize=(7, 8))

# Standard stacked bars: each row spans [0, 1]
left_masc = torch.zeros_like(masc)
left_neutral = masc
left_fem = masc + neutral

ax.barh(y, masc, left=left_masc, height=0.75, label="Masculine effect")
ax.barh(y, neutral, left=left_neutral, height=0.75, label="Neutral")
ax.barh(y, fem, left=left_fem, height=0.75, label="Feminine effect")

ax.set_yticks(y)
ax.set_yticklabels(names)

ax.set_xlim(0, 1)
ax.set_xlabel("Fraction of samples")
ax.set_title(f"Thresholded gender sensitivity by class (CAV acc: {accuracy:.3f}) (biased model)")

ax.grid(axis="x", alpha=0.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.4, -0.08), ncol=3, fontsize=14)

plt.tight_layout()
plt.show()

## 3. Unsupervised concepts analysis

### 3.1 Splitting the model and computing activations

Already done in section 2.1.

### 3.2 Learn concepts

We use the SemiNMF (a relaxed version of the Non-negative Matrix Factorization) to learn the concepts. It decomposes activations $A = enc(X)$ into a product of concept activations $Z$ and concept vectors $W$:

$$
A = ZW
$$

Please rely on [`SemiNMFConcepts`](https://for-sight-ai.github.io/interpreto/api/concepts/concept_spaces/optim/#interpreto.concepts.SemiNMFConcepts).

In [ ]:
from interpreto.concepts import SemiNMFConcepts

# instantiate from the splitter
concept_explainer = SemiNMFConcepts(splitter, nb_concepts=50, device=device)

# learn concepts as patterns in the activations
concept_explainer.fit(train_acts.to(device), max_iter=500)

### 3.3 Interpret concepts

Concepts are currently abstract directions in the activation space, e need to assign them meaning. For each concept $cpt$, we compute the topk activating words, said differently, the words which aligns the most to the concept direction in the activation space.

$$
topk(cpt) = topk_X\ enc(x) \cdot W_{cpt}
$$

You can use [`TopKInputs`](https://for-sight-ai.github.io/interpreto/api/concepts/interpretations/topk_inputs/).

In [ ]:
from interpreto.concepts.interpretations import TopKInputs

topk = TopKInputs(
    concept_explainer=concept_explainer,
    k=5,
    use_unique_words=3,  # extend up to 3-words ngrams
    unique_words_kwargs={
        "count_min_threshold": max(1, round(len(test_inputs) * 0.002)),  # present in at least 0.2% of the inputs
        "lemmatize": True,
    },
)
interpretations = topk.interpret(inputs=test_inputs)
interpretations = {str(k): list(v.keys()) for k, v in interpretations.items()}

### 3.4 Estimate concepts importance

Concepts might be present but not relevant for the predictions. This step evaluates the global importance of concepts with respect to each class.

The importance is given by the average (over samples) of the concepts to outputs gradients weighted by the concepts activations:
$$
mean(\nabla_z cls(ZW) \cdot Z)
$$

Rely on the explainer's [`concept_output_gradients`](https://for-sight-ai.github.io/interpreto/api/concepts/concept_spaces/optim/#interpreto.concepts.methods.DictionaryLearningExplainer.concept_output_gradient) method.

In [ ]:
gradients_list = concept_explainer.concept_output_gradient(
    test_acts, targets=None, tqdm_bar=True, normalization=False, concepts_x_gradients=True
)
importances = torch.stack(gradients_list).squeeze().mean(0).cpu()  # (cls, cpt)

### 3.5 Visualize concepts

You need to click on the classes names to see the relevant concepts.

In [ ]:
from interpreto import plot_concepts

plot_concepts(classes_names=classes_names, concepts_importances=importances, concepts_labels=interpretations, top_k=5)